# DSP Series - Notebook 1: Foundations of Noise & Time-Domain Filtering
**Anchored in Chapter 2, 3, & 15 of *The Scientist and Engineer's Guide to DSP* by Dr. Steven W. Smith**

In a pure control systems textbook, sensors are perfect. In the real world, a marine engineering pipeline is swimming in noise. If you hook a raw, unfiltered Inertial Measurement Unit (IMU) straight to an autopilot, the high-frequency garbage will destroy your machinery. 

This notebook assumes **zero prior experience** with signal processing. We will build an intuitive understanding of sensor noise and implement our very first time-domain filter.

---

## Section 1: The Physics of Sensor Noise

When a vessel is underway, an IMU measuring heading ($\psi$) or yaw rate ($r$) experiences two distinct profiles of data:
1. **The Signal:** The low-frequency, large-amplitude physical maneuvering of the hull (the actual path of the boat).
2. **The Noise:** High-frequency, low-amplitude disturbances. This comes from structural engine vibrations, shaft hum, electrical line interference, and wave impacts slamming the hull.

### The Engineering Penalty: Actuator Abuse & "Chaffing"
Students often think noise just makes a graph look ugly. In marine engineering, the penalty is mechanical destruction. 

An autopilot uses a **Derivative ($D$)** loop to check turns. Mathematically, derivative action calculates the *rate of change* (slope) of the sensor signal. Because high-frequency noise spikes up and down incredibly fast, its slope is massive. 

If you feed raw noise into a PID loop, the controller amplifies these spikes and commands the rudder actuators to micro-correct left and right hundreds of times a minute. This is called **rudder chaffing**. The rudder cannot move fast enough to steer the ship at 10 Hz, so this energy is completely wasted—rapidly overheating hydraulic steering pumps, wearing out mechanical linkages, and burning excessive fuel.

Let's generate a realistic, noisy marine sensor stream to visualize this hardware threat.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import VBox, HBox
%matplotlib inline

# Configure clean plotting defaults
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

def generate_marine_data(noise_amplitude):
    """Generates a clean hull maneuver corrupted by structural noise."""
    t = np.linspace(0, 10, 1000) # 10 seconds of data sampled at 100 Hz
    dt = t[1] - t[0]
    
    # True underlying ship behavior (Low-frequency turn)
    true_heading = 45.0 * np.sin(2 * np.pi * 0.05 * t) 
    
    # Structural/Engine Noise (High-frequency variance)
    np.random.seed(42) 
    engine_noise = noise_amplitude * np.sin(2 * np.pi * 8.0 * t) 
    white_noise = (noise_amplitude * 0.5) * np.random.randn(len(t))
    
    noisy_heading = true_heading + engine_noise + white_noise
    
    # Calculate approximate "Rudder Command Effort" via derivative
    rudder_effort = np.diff(noisy_heading, prepend=noisy_heading[0]) / dt
    
    fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    
    axs[0].plot(t, noisy_heading, 'r', alpha=0.6, label='Raw Noisy IMU Heading')
    axs[0].plot(t, true_heading, 'k-', linewidth=2, label='True Hull Path')
    axs[0].set_ylabel("Heading [deg]")
    axs[0].set_title("The Reality of Raw Marine Sensor Streams")
    axs[0].legend(loc='upper right')
    
    # Plot the derivative consequence (Actuator Stress)
    axs[1].plot(t, rudder_effort, 'orange', alpha=0.7)
    axs[1].set_ylabel("Actuator Stress Rate [deg/s]")
    axs[1].set_xlabel("Time [s]")
    axs[1].set_title("Resulting Rudder Hydraulic Demand (Actuator Chaffing)")
    
    plt.tight_layout()
    plt.show()

# Interactive slider to amplify engine/structural vibrations
widgets.interact(generate_marine_data, noise_amplitude=(0.0, 5.0, 0.2))

interactive(children=(FloatSlider(value=2.4000000000000004, description='noise_amplitude', max=5.0, step=0.2),…

<function __main__.generate_marine_data(noise_amplitude)>

## Section 2: The Moving Average Filter (Smith Chapter 15)

To save our steering gear, we must strip away the high-frequency chatter while preserving the underlying hull maneuver. Smith introduces the **Moving Average Filter** as the premier tool for time-domain noise reduction. It is incredibly simple, highly intuitive, and requires zero advanced signal processing background.

### How It Works
The moving average operates by taking a group of consecutive samples, averaging them together, and using that average value to represent the central data point. Mathematically, for a window side length of $M$ points, the filter output $y[i]$ is defined as:

$$y[i] = \frac{1}{M} \sum_{j=0}^{M-1} x[i + j]$$

In plain English: a window slider passes over the array. If the window size is 5, it adds up 5 consecutive points and divides by 5. Because high-frequency noise randomly jumps above and below the true line, adding them together causes them to naturally cancel each other out, leaving the clean, steady baseline behind.

In [2]:
def apply_moving_average(window_size):
    # Re-generate our standard baseline noisy boat data (100 Hz sampling rate)
    t = np.linspace(0, 10, 1000)
    dt = t[1] - t[0]
    
    # True underlying ship behavior (Low-frequency turn)
    true_heading = 45.0 * np.sin(2 * np.pi * 0.05 * t)
    
    np.random.seed(42)
    engine_noise = 2.5 * np.sin(2 * np.pi * 8.0 * t)
    white_noise = 1.0 * np.random.randn(len(t))
    noisy_heading = true_heading + engine_noise + white_noise
    
    # 1. Calculate the ideal, clean actuator demand (The perfect sine wave baseline)
    true_rudder_effort = np.diff(true_heading, prepend=true_heading[0]) / dt
    
    # 2. Calculate baseline raw noisy rudder chaffing for comparison
    raw_rudder_effort = np.diff(noisy_heading, prepend=noisy_heading[0]) / dt
    
    # 3. Apply the Moving Average Filter array loop (Smith Ch 15)
    filtered_heading = np.zeros_like(noisy_heading)
    half_win = window_size // 2
    for i in range(len(noisy_heading)):
        start_idx = max(0, i - half_win)
        end_idx = min(len(noisy_heading), i + half_win + 1)
        filtered_heading[i] = np.mean(noisy_heading[start_idx:end_idx])
        
    # 4. Calculate Actuator Stress on the filtered telemetry signal
    cleaned_rudder_effort = np.diff(filtered_heading, prepend=filtered_heading[0]) / dt
    
    # Create the two stacked comparison plots to explicitly show the hydraulic payoff
    fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    
    # Top Plot: Heading Signals
    axs[0].plot(t, noisy_heading, 'r', alpha=0.2, label='Raw Noisy Telemetry Input')
    axs[0].plot(t, true_heading, 'k--', alpha=0.7, label='True Path (No Noise)')
    axs[0].plot(t, filtered_heading, 'b-', linewidth=2.5, label=f'Filtered Signal (Window M={window_size})')
    axs[0].set_ylabel("Heading [deg]")
    axs[0].set_title("Time-Domain Smoothing via a Moving Average Window")
    axs[0].legend(loc='upper right')
    
    # Bottom Plot: Actuator Chaffing Consequence (Explicitly showing reduction)
    axs[1].plot(t, raw_rudder_effort, 'orange', alpha=0.15, label='Raw Actuator Stress (Chaffing)')
    axs[1].plot(t, true_rudder_effort, 'k--', alpha=0.7, label='True Idealized Rudder Demand (Sine Wave)')
    axs[1].plot(t, cleaned_rudder_effort, 'g-', linewidth=2, label='Filtered Actuator Stress')
    axs[1].set_ylabel("Actuator Stress Rate [deg/s]")
    axs[1].set_xlabel("Time [s]")
    axs[1].set_title("The Mechanical Payoff: Recovering the True Control Signal")
    axs[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Slider must be an odd number to maintain symmetric window centering
widgets.interact(apply_moving_average, 
                 window_size=widgets.IntSlider(value=11, min=3, max=101, step=2, description='Window Size M:'))

interactive(children=(IntSlider(value=11, description='Window Size M:', max=101, min=3, step=2), Output()), _d…

<function __main__.apply_moving_average(window_size)>

## Section 3: The Architectural Trap: Centered vs. Trailing Windows

Now we face a vital engineering decision: **Where do we place our window relative to the current time index?** How we answer this dictates whether our filter can actually be run on a live ship.

### 1. The Centered Window (Non-Causal Post-Processing)
A **Centered Moving Average** splits its window symmetrically across the data point. To filter point $i$, it reaches backward into the past, handles point $i$, and reaches *forward into the future* to gather points ($i+1, i+2$).

CENTERED WINDOW (Looking into the future):
[ x[i-2] ]  [ x[i-1] ]  [  x[i]  ]  [ x[i+1] ]  [ x[i+2] ] ──► Filtered output y[i]
▲
Current Point


* **The Massive Benefit:** Because it averages data symmetrically from both sides, it eliminates high-frequency noise **without shifting the signal in time**. It introduces **zero phase lag**.
* **The Catch:** It requires information from the future. It is **non-causal**. If you are sitting at your desk running diagnostics on a pre-recorded `.csv` log file from last week's sea trials, this is your gold standard tool.

### 2. The Trailing Window (Causal Real-Time Control)
When standing watch or running autopilot firmware inside an embedded flight computer, **you cannot predict the future**. You do not have access to sample $x[i+1]$ because that moment in time hasn't happened yet. You can only look backward.

TRAILING WINDOW (Real-time history only):
[ x[i-4] ]  [ x[i-3] ]  [ x[i-2] ]  [ x[i-1] ]  [  x[i]  ] ──► Filtered output y[i]
▲
Current Time


* **The Reality:** This is the only window placement that can physically execute in real-time (**causal**).
* **The Structural Penalty:** Because the window is weighted strictly toward the past, it introduces a severe geometric **Phase Lag (Delay)** proportional to the window size. 

Let's look at a head-to-head comparison to see exactly how these geometric choices deform

In [3]:
def compare_window_geometries(window_size):
    t = np.linspace(0, 10, 1000)
    dt = t[1] - t[0]
    
    true_heading = 45.0 * np.sin(2 * np.pi * 0.05 * t)
    np.random.seed(42)
    noisy_heading = true_heading + 2.5 * np.sin(2 * np.pi * 8.0 * t) + 1.0 * np.random.randn(len(t))
    
    # Base ideal actuator demand
    true_rudder_effort = np.diff(true_heading, prepend=true_heading[0]) / dt
    raw_rudder_effort = np.diff(noisy_heading, prepend=noisy_heading[0]) / dt
    
    # Setup arrays
    centered_heading = np.zeros_like(noisy_heading)
    trailing_heading = np.zeros_like(noisy_heading)
    
    half_win = window_size // 2
    
    for i in range(len(noisy_heading)):
        # --- 1. CENTERED WINDOW CALCULATIONS (Post-Processing) ---
        c_start = max(0, i - half_win)
        c_end = min(len(noisy_heading), i + half_win + 1)
        centered_heading[i] = np.mean(noisy_heading[c_start:c_end])
        
        # --- 2. TRAILING WINDOW CALCULATIONS (Real-Time Causal) ---
        t_start = max(0, i - window_size + 1)
        trailing_heading[i] = np.mean(noisy_heading[t_start:i+1])
        
    # Calculate resultant rudder demands from both filtering philosophies
    centered_rudder = np.diff(centered_heading, prepend=centered_heading[0]) / dt
    trailing_rudder = np.diff(trailing_heading, prepend=trailing_heading[0]) / dt
    
    fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    
    # Top Plot: Geometry Heading Lag Comparison
    axs[0].plot(t, true_heading, 'k--', alpha=0.8, label='True Path')
    axs[0].plot(t, centered_heading, 'cyan', linewidth=2, label='Centered (Zero Phase Lag)')
    axs[0].plot(t, trailing_heading, 'magenta', linewidth=2, label='Trailing (Real-Time Lag)')
    axs[0].set_ylabel("Heading [deg]")
    axs[0].set_title("Geometric Impact of Window Placement on Phase Shift")
    axs[0].legend(loc='upper right')
    
    # Bottom Plot: Hydraulic Demand Consequence
    axs[1].plot(t, true_rudder_effort, 'k--', alpha=0.8, label='True Ideal Sine Demand')
    axs[1].plot(t, centered_rudder, 'cyan', linewidth=2, label='Centered Filter Demand')
    axs[1].plot(t, trailing_rudder, 'magenta', linewidth=2, label='Trailing Filter Demand')
    axs[1].set_ylabel("Actuator Stress Rate [deg/s]")
    axs[1].set_xlabel("Time [s]")
    axs[1].set_title("The Resulting Demand Shifts On Steering Gear")
    axs[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

widgets.interact(compare_window_geometries, 
                 window_size=widgets.IntSlider(value=21, min=3, max=101, step=2, description='Window Size M:'))

interactive(children=(IntSlider(value=21, description='Window Size M:', max=101, min=3, step=2), Output()), _d…

<function __main__.compare_window_geometries(window_size)>

## Section 4: The Closed-Loop Failure: Autopilot "Hunting"

When you use a large window size in Section 3, both filters clear out the random noise and compress the high-frequency hydraulic stress. However, look closely at the **magenta (Trailing)** line. Its peaks are shifted significantly behind the true timeline.

### Why This Destabilizes the Ship (The Maneuvering Crisis)
Because an embedded flight computer is restricted to using a Trailing window, increasing the window size to get cleaner data directly pumps time delay ($T_{\text{delay}}$) into the feedback system.

* **The Consequence:** The autopilot reads where the boat *was* 200 milliseconds ago. If the bow is rapidly swinging past its target heading, the filtered telemetry tells the flight computer that everything is fine. 
* **The "Hunting" Loop:** By the time the filter updates and the autopilot panics, shouting *Counter-Rudder!*, the ship has drastically overshot. The autopilot slams the rudder hard over, but that response is *also* delayed by the filter. The system begins frantically chasing its own tail—a disastrous control anomaly known as **hunting oscillations**.

Let's look at how our causal trailing average performs dynamically inside an active heading loop.

In [4]:
def simulate_closed_loop_steering(window_size, autopilot_gain):
    t = np.linspace(0, 15, 1500)
    dt = t[1] - t[0]
    
    # Target Heading Setpoint: Step command to 30 degrees at t = 1s
    setpoint = np.zeros_like(t)
    setpoint[t >= 1.0] = 30.0
    
    # Ship Plant Emulation arrays
    actual_heading = np.zeros_like(t)
    filtered_heading = np.zeros_like(t)
    yaw_rate = 0.0
    
    # Simulated vessel time constant (Inertia lag of a standard hull)
    T_hull = 1.5 
    K_hull = 0.4
    
    # Step through time to close the feedback loop dynamically (Strictly Causal)
    for i in range(1, len(t)):
        # 1. Simulate Noisy Sensor Acquisition
        np.random.seed(i)
        noise = 1.5 * np.sin(2 * np.pi * 8.0 * t[i]) + 0.5 * np.random.randn()
        raw_sensor_reading = actual_heading[i-1] + noise
        
        # 2. Apply Causal Trailing Moving Average Filter historic slice
        start = max(0, i - window_size + 1)
        history = actual_heading[start:i] + 1.5 * np.sin(2 * np.pi * 8.0 * t[start:i])
        filtered_heading[i] = np.mean(history) if len(history) > 0 else raw_sensor_reading
        
        # 3. Autopilot Logic based on FILTERED telemetry
        error = setpoint[i] - filtered_heading[i]
        rudder_command = autopilot_gain * error
        rudder_command = np.clip(rudder_command, -35.0, 35.0)
        
        # 4. Hull Physics Integration (1st-Order Nomoto Step)
        yaw_rate_dot = (K_hull * rudder_command - yaw_rate) / T_hull
        yaw_rate += yaw_rate_dot * dt
        actual_heading[i] = actual_heading[i-1] + yaw_rate * dt

    plt.figure(figsize=(11, 5))
    plt.plot(t, setpoint, 'k--', label='Desired Heading Setpoint', linewidth=2)
    plt.plot(t, actual_heading, 'b-', linewidth=2.5, label='Actual Ship Trajectory')
    plt.title("The System Consequence of Causal Phase Lag: Autopilot 'Hunting' Meltdown")
    plt.ylabel("Heading [deg]")
    plt.xlabel("Time [s]")
    plt.legend(loc='lower right')
    plt.ylim(-20, 60)
    plt.show()

# Interactively balance filter smoothness against loop stability
widgets.interact(simulate_closed_loop_steering, 
                 window_size=widgets.IntSlider(value=5, min=1, max=61, step=2, description='Filter Window:'),
                 autopilot_gain=widgets.FloatSlider(value=1.5, min=0.5, max=5.0, step=0.1, description='Autopilot Gain:'))

interactive(children=(IntSlider(value=5, description='Filter Window:', max=61, min=1, step=2), FloatSlider(val…

<function __main__.simulate_closed_loop_steering(window_size, autopilot_gain)>

## Section 5: The Exponential Moving Average (EMA) & The "Cold Start" Flushing Rule

Now that we understand the limits of traditional window filters, we can look at how production-grade embedded GNC systems handle real-time noise reduction. In firmware engineering, we often abandon standard window filters entirely in favor of recursive structures.

### 1. The Exponential Moving Average (EMA)
The standard moving average is a "Finite Impulse Response" (FIR) filter—it drops old data completely off a cliff the moment it exits the trailing window array. Steven W. Smith covers the alternative in Chapter 19 as the **Single-Pole Low-Pass Filter**. 

Instead of tracking a massive historic array in memory, the EMA calculates the current filtered point ($y[i]$) using only two numbers: the current raw sensor reading ($x[i]$) and the *previous* filtered value ($y[i-1]$).

$$y[i] = \alpha \cdot x[i] + (1 - \alpha) \cdot y[i-1]$$

Where $\alpha$ (alpha) is a smoothing coefficient between $0.0$ and $1.0$. 

* **The Embedded Benefit:** It requires zero historical arrays. The microcontroller only needs to store *one* float variable in memory (`y_previous`), making it incredibly lightweight for real-time operating systems (RTOS).
* **The GNC Payoff:** Because the mathematical weights decay exponentially into the past, the EMA naturally cuts down on phase lag compared to a standard trailing window that delivers equivalent noise reduction. It heavily biases the tracking loop toward the present moment.

---

### 2. The Danger of a "Cold Start" (Why Filters Must Be Flushed)
"Flushing" or initializing the filter addresses what happens the exact millisecond you boot up the computer, toggle the autopilot from manual to automatic, or recover from a temporary sensor drop-out.

If your code defaults to initializing the previous filter memory to zero (`y_prev = 0.0`), and your ship is currently sitting at a steady cruising heading of $45^\circ$, the very first iteration of your EMA filter (assuming $\alpha = 0.1$) will compute:

$$y[0] = 0.1 \cdot (45.0) + 0.9 \cdot (0.0) = 4.5^\circ$$

The filter thinks the ship is at $4.5^\circ$ instead of $45^\circ$. It will take dozens of samples for the filter to slowly ramp up and "climb" to reality. 



* **The System Meltdown:** During those few seconds of ramping up, the autopilot sees a massive, fake error ($45^\circ - 4.5^\circ = 40.5^\circ$ error). It will violently slam the rudder hard over to correct a deviation that doesn't actually exist. You can easily cause a physical steering accident right at the moment you flip the switch to automatic.
* **The Engineering Solution:** "Flushing" means that on the very first sample ($t=0$), or during a reset, you completely bypass the averaging math and force the filter's memory to match the raw sensor input exactly: `y_prev = x[0]`. This pre-loads or "flushes" the historical lag out of the system, ensuring the very first filtered output matches reality instantly.

Let's look at the final code demo to see the head-to-head comparison between standard windows and the EMA, and watch what happens when an unflushed filter tricks our actuator logic.

In [5]:
def demonstrate_ema_and_flushing(alpha, filter_window, cold_start):
    # Setup a 10-second marine simulation sampled at 100 Hz
    t = np.linspace(0, 10, 1000)
    dt = t[1] - t[0]
    
    # Boat is steady at 45 degrees heading, then begins a gentle turn at t=3s
    true_heading = np.ones_like(t) * 45.0
    true_heading[t > 3.0] = 45.0 + 15.0 * np.sin(2 * np.pi * 0.05 * (t[t > 3.0] - 3.0))
    
    np.random.seed(42)
    noisy_heading = true_heading + 2.0 * np.sin(2 * np.pi * 8.0 * t) + 0.5 * np.random.randn(len(t))
    
    # Setup arrays for outputs
    standard_filtered = np.zeros_like(noisy_heading)
    ema_filtered = np.zeros_like(noisy_heading)
    
    # 1. Standard Moving Average Window Loop
    half_win = filter_window // 2
    for i in range(len(noisy_heading)):
        start_idx = max(0, i - half_win)
        end_idx = min(len(noisy_heading), i + half_win + 1)
        standard_filtered[i] = np.mean(noisy_heading[start_idx:end_idx])
        
    # 2. Exponential Moving Average Loop with Initial Condition Logic
    if cold_start:
        # The Dangerous Assumption: Initializing memory blindly to 0.0
        y_prev = 0.0 
    else:
        # The Pro Solution: "Flushing" the filter memory with the first real sample
        y_prev = noisy_heading[0]
        
    for i in range(len(noisy_heading)):
        y_current = alpha * noisy_heading[i] + (1.0 - alpha) * y_prev
        ema_filtered[i] = y_current
        y_prev = y_current
        
    # Calculate resultant rudder steering demands (approximate derivatives)
    ema_rudder_effort = np.diff(ema_filtered, prepend=ema_filtered[0]) / dt
    
    fig, axs = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    
    # Top Plot: Telemetry Tracking
    axs[0].plot(t, true_heading, 'k--', alpha=0.8, label='True Hull Heading')
    axs[0].plot(t, standard_filtered, 'b-', alpha=0.5, label=f'Standard Window (M={filter_window})')
    axs[0].plot(t, ema_filtered, 'g-', linewidth=2, label=f'EMA (alpha={alpha})')
    axs[0].set_ylabel("Heading [deg]")
    axs[0].set_title("Standard Moving Average vs. Exponential Moving Average (EMA)")
    axs[0].legend(loc='lower right')
    
    if cold_start:
        axs[0].annotate('Dangerous Cold Start Ramp-up!', xy=(0.2, 15), xytext=(1.5, 10),
                    arrowprops=dict(facecolor='red', shrink=0.05))
                    
    # Bottom Plot: The Rudder Actuator Response
    axs[1].plot(t, ema_rudder_effort, 'g-', linewidth=2, label='EMA Filtered Actuator Demand')
    axs[1].set_ylabel("Actuator Stress Rate [deg/s]")
    axs[1].set_xlabel("Time [s]")
    axs[1].set_title("The Resulting Actuator Stress Profile")
    axs[1].legend(loc='upper right')
    
    if cold_start:
         axs[1].set_ylim(-150, 150) # Scale out to catch the massive initial startup surge
    else:
         axs[1].set_ylim(-30, 30)
         
    plt.tight_layout()
    plt.show()

widgets.interact(demonstrate_ema_and_flushing, 
                 alpha=widgets.FloatSlider(value=0.08, min=0.01, max=0.5, step=0.01, description='EMA Alpha:'),
                 filter_window=widgets.IntSlider(value=25, min=3, max=101, step=2, description='Std Window M:'),
                 cold_start=widgets.Checkbox(value=False, description='Enable Cold Start (No Flush)'))

interactive(children=(FloatSlider(value=0.08, description='EMA Alpha:', max=0.5, min=0.01, step=0.01), IntSlid…

<function __main__.demonstrate_ema_and_flushing(alpha, filter_window, cold_start)>

## Summary: The GNC Engineer's Balancing Act

You have just witnessed the foundational paradox of signal processing within an embedded control system loop:

1. **Too Little Filtering (Small Window):** Data is instantaneous (no phase lag), but structural engine chatter passes straight through, destroying your steering pump hydraulics via **actuator chaffing**.
2. **Too Much Filtering (Large Window):** The noise is beautifully flat, but the resulting **causal trailing phase lag** forces the autopilot to live in the past. The loop loses its stability margins, causing the ship to hunt and violently sway.
3. **Improper Initialization:** Failing to **flush** your recursive filters on a cold start introduces massive, artificial error vectors that spike your actuators and can cause immediate physical accidents.

### What's Next?
We need a smarter tool. A tool that can aggressively chop away high-frequency wave noise and engine hum *without* needing a massive historical trailing window that delays our time response. 

To build that tool, we have to stop looking at data as lines over time, and start looking at data through its frequencies. In the next notebook, we will unwrap Steven W. Smith’s guide to the **Frequency Domain and the Fast Fourier Transform (FFT)** to see exactly how to select the perfect filter cutoff frequency mathematically.